# 07 · Sesión 2 · Validación cruzada y outliers

**Duración:** 2h 30min  (parte teórica)
**Después:** práctica de 1h 30min en `07_practica_final.ipynb`

## ¿Qué vas a entender al terminar?

1. Por qué **una sola partición train/test puede no ser suficiente**.
2. Qué es la **validación cruzada** y cómo se interpreta.
3. Qué es un **outlier** y cómo detectarlo con la regla del **IQR**.
4. Cómo afectan los outliers a un modelo de regresión.

---

## 1 · Repaso y limitación del split único

En la sesión 1 usamos una sola partición train/test. Eso está bien para empezar, pero **el resultado depende del corte que haya tocado**.

### Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes, make_regression
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
data = load_diabetes(as_frame=True)
df = data.frame.copy()
X = df.drop(columns='target')
y = df['target']

### Un mismo modelo, distintos resultados

Entrenamos el mismo `LinearRegression` cinco veces cambiando solo el `random_state` del split. Apuntamos el R² cada vez.

In [ ]:
resultados = []

for seed in [1, 7, 21, 42, 99]:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed
    )
    model = LinearRegression().fit(X_train, y_train)
    pred = model.predict(X_test)
    resultados.append({
        'seed': seed,
        'R2': round(r2_score(y_test, pred), 3),
        'MAE': round(mean_absolute_error(y_test, pred), 2),
    })

pd.DataFrame(resultados)

> **Idea clave:** el modelo es el mismo, pero el score cambia según la partición. Cuando esto pasa, no sabemos cuánto del score es **señal real** y cuánto es **azar del corte**. Necesitamos una evaluación más estable.

---

## 2 · Validación cruzada (K-Fold)

En lugar de un solo corte, hacemos **varios**. Dividimos los datos en K trozos (*folds*). Entrenamos K veces: cada vez, un fold distinto es el test y el resto train.

```
fold 1: [TEST][train][train][train][train]
fold 2: [train][TEST][train][train][train]
...
fold 5: [train][train][train][train][TEST]
```

Obtenemos K scores. Reportamos la **media** (rendimiento esperado) y la **desviación típica** (qué tan estable es).

In [ ]:
model = LinearRegression()
cv_scores = cross_val_score(model, X, y, cv=5, scoring='r2')

print('Scores R² por fold :', np.round(cv_scores, 3))
print(f'Media              : {cv_scores.mean():.3f}')
print(f'Desv. típica       : {cv_scores.std():.3f}')

> **Cómo leerlo:** si la media es 0.45 y la desv. típica es 0.05, esperamos un R² "alrededor de 0.45" en datos nuevos, con una incertidumbre de ±0.05. Si la desv. típica fuera 0.20, la cosa está mucho menos clara.

### Ejercicio 1 · Validación cruzada con MAE

Haz `cross_val_score` con la métrica de error absoluto medio.  
**Pista:** en scikit-learn los errores se devuelven con signo negativo (porque sklearn sigue la convención de "mayor es mejor").

In [ ]:
# Tu código aquí

**Solución**

In [ ]:
cv_mae = cross_val_score(model, X, y, cv=5, scoring='neg_mean_absolute_error')
print('MAE por fold:', np.round(-cv_mae, 2))
print(f'MAE medio   : {(-cv_mae).mean():.2f}')

---

## 3 · ¿Qué es un outlier?

Un outlier es un dato **muy alejado del comportamiento habitual**. Puede ser:
- un error de medida;
- un caso raro pero real (un piso de 500 m² en una calle de pisos de 80 m²);
- un valor extremo que **tira mucho** del modelo.

### Lo vemos con un ejemplo
Dataset pequeño y limpio → añadimos 5 outliers a mano.

In [ ]:
X_base, y_base = make_regression(
    n_samples=120, n_features=1, noise=15, random_state=10
)

X_out = X_base.copy()
y_out = y_base.copy()
X_out[:5] = np.array([[2.8], [3.0], [3.2], [3.4], [3.6]])
y_out[:5] = np.array([280, 310, 330, 350, 370])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].scatter(X_base, y_base, alpha=0.7, color='steelblue')
axes[0].set_title('Dataset limpio')

axes[1].scatter(X_out, y_out, alpha=0.7, color='steelblue')
axes[1].scatter(X_out[:5], y_out[:5], color='red', s=80, label='outliers')
axes[1].set_title('Mismo dataset + 5 outliers')
axes[1].legend()

for ax in axes:
    ax.set_xlabel('X')
    ax.set_ylabel('y')
plt.tight_layout()
plt.show()

---

## 4 · Efecto de los outliers en el modelo

Una regresión lineal busca la recta que **mejor se ajusta en promedio**. Si hay puntos muy alejados, **tiran de la recta hacia ellos** y dejan de representar bien a la mayoría.

In [ ]:
model_base = LinearRegression().fit(X_base, y_base)
model_out  = LinearRegression().fit(X_out,  y_out)

x_line = np.linspace(
    min(X_out.min(), X_base.min()),
    max(X_out.max(), X_base.max()),
    100
).reshape(-1, 1)

plt.figure(figsize=(8, 5))
plt.scatter(X_base, y_base, alpha=0.5, label='datos limpios')
plt.scatter(X_out[:5], y_out[:5], color='red', s=80, label='outliers')
plt.plot(x_line, model_base.predict(x_line), color='green',
         linewidth=2, label='recta SIN outliers')
plt.plot(x_line, model_out.predict(x_line),  color='orange',
         linewidth=2, label='recta CON outliers')
plt.xlabel('X')
plt.ylabel('y')
plt.title('Cómo los outliers tiran de la recta')
plt.legend()
plt.show()

### Ejercicio 2 · Describe con tus palabras

1. ¿Hacia dónde se ha movido la recta al añadir los outliers?
2. ¿Representa mejor o peor a la mayoría de los datos?
3. ¿Crees que el modelo sigue siendo útil?

In [ ]:
# Tu código aquí

---

## 5 · Detectar outliers con IQR

Una regla simple y muy usada:

- Calcula el primer y tercer cuartil: **Q1** y **Q3**.
- **IQR = Q3 − Q1** (rango intercuartílico).
- Marca como outlier todo lo que queda fuera de `[Q1 − 1.5·IQR, Q3 + 1.5·IQR]`.

In [ ]:
df_out = pd.DataFrame({'x': X_out.ravel(), 'y': y_out})

q1 = df_out['y'].quantile(0.25)
q3 = df_out['y'].quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

mask = (df_out['y'] >= lower) & (df_out['y'] <= upper)

print(f'Q1 = {q1:.2f}   Q3 = {q3:.2f}   IQR = {iqr:.2f}')
print(f'Límite inferior: {lower:.2f}')
print(f'Límite superior: {upper:.2f}')
print()
print(f'Sin outliers : {mask.sum()} filas')
print(f'Outliers     : {(~mask).sum()} filas')

### Ejercicio 3 · Limpia y reentrena

1. Crea `df_limpio` filtrando las filas sin outliers.
2. Entrena una nueva regresión sobre `df_limpio`.
3. Dibuja las dos rectas (con y sin outliers) sobre los datos limpios y los outliers.
4. ¿Ha cambiado la pendiente? ¿Te parece más representativa?

In [ ]:
# Tu código aquí

**Solución**

In [ ]:
df_limpio = df_out[mask].copy()

X_clean = df_limpio[['x']]
y_clean = df_limpio['y']

model_clean = LinearRegression().fit(X_clean, y_clean)

plt.figure(figsize=(8, 5))
plt.scatter(df_limpio['x'], df_limpio['y'], alpha=0.7, label='datos limpios')
plt.scatter(df_out[~mask]['x'], df_out[~mask]['y'],
            color='red', s=80, label='outliers (no usados)')
plt.plot(x_line, model_out.predict(x_line),   color='orange',
         linewidth=2, label='modelo con outliers')
plt.plot(x_line, model_clean.predict(x_line), color='green',
         linewidth=2, label='modelo limpio')
plt.xlabel('X')
plt.ylabel('y')
plt.title('Comparación final')
plt.legend()
plt.show()

---

## 6 · Mini-práctica guiada

**Sobre el dataset de diabetes** (`load_diabetes`):

1. Calcula el R² con `cross_val_score(cv=5)` y apúntalo.
2. Detecta outliers en la columna `target` usando IQR. ¿Cuántos hay?
3. Crea `df_diabetes_limpio` sin esos outliers.
4. Vuelve a calcular el R² por cross-validation sobre el dataset limpio.
5. **Compara:** ¿ha subido, ha bajado o se ha quedado igual? ¿Por qué crees que pasa eso?

Escribe tus respuestas en la celda de abajo (en comentarios o en Markdown).

In [ ]:
# Tu código aquí

---

## Cierre

| Concepto | Para qué sirve |
|---|---|
| `cross_val_score` | Obtener una evaluación **estable** del modelo, no dependiendo de un solo corte. |
| Outliers (IQR) | Detectar valores extremos que pueden **falsear** el modelo. |

**Mensaje para llevar a casa:**

- *Datos limpios > modelo complejo.* A veces mejorar la calidad de los datos da más resultado que cambiar el algoritmo.
- *Validar bien importa.* Un R² bonito con un solo split puede no significar nada.
- *Borrar outliers tiene coste.* A veces son errores, a veces son los casos más interesantes. Siempre hay que investigarlos antes de quitarlos.